In [1]:
import sys
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils.hospital_env import HospitalEnv

data = pd.read_csv("../data/synthetic_hospital_data.csv")

In [2]:
STATE_DIM = 5
ACTION_DIM = 2

def normalize_state(state):
    arrival, slot, priority, no_show, icu_ratio = state
    return np.array([
        arrival / 480.0,
        slot / 480.0,
        priority / 2.0,
        no_show,
        icu_ratio
    ], dtype=np.float32)

In [3]:
class ReplayBuffer:
    def __init__(self, capacity=5000):
        self.buffer = deque(maxlen=capacity)

    def push(self, s, a, r, ns, d):
        self.buffer.append((s, a, r, ns, d))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (
            np.array(s),
            np.array(a),
            np.array(r),
            np.array(ns),
            np.array(d)
        )

    def __len__(self):
        return len(self.buffer)

In [4]:
class DQN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(STATE_DIM, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, ACTION_DIM)
        )

    def forward(self, x):
        return self.net(x)

In [5]:
policy_net = DQN()
target_net = DQN()

target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=0.001)
memory = ReplayBuffer()

episodes = 200
batch_size = 64
gamma = 0.99

epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995

target_update = 5

In [6]:
episode_rewards = []
episode_waiting = []
episode_util = []

for ep in range(episodes):

    env = HospitalEnv(data)
    state = normalize_state(env.reset())
    done = False

    total_reward = 0
    total_wait = 0
    total_util = 0
    steps = 0

    while not done:

        if random.random() < epsilon:
            action = random.randint(0, ACTION_DIM - 1)
        else:
            with torch.no_grad():
                q_vals = policy_net(torch.tensor(state, dtype=torch.float32).unsqueeze(0))
                action = torch.argmax(q_vals, dim=1).item()

        next_state, reward, done, info = env.step(action)

        if not done:
            next_state_n = normalize_state(next_state)
        else:
            next_state_n = np.zeros(STATE_DIM, dtype=np.float32)

        memory.push(state, action, reward, next_state_n, done)

        state = next_state_n
        total_reward += reward
        total_wait += info["waiting_time"]
        total_util += info["resource_util"]
        steps += 1

        if len(memory) >= batch_size:

            s, a, r, ns, d = memory.sample(batch_size)

            s  = torch.tensor(s, dtype=torch.float32)
            ns = torch.tensor(ns, dtype=torch.float32)
            a  = torch.tensor(a, dtype=torch.long)
            r  = torch.tensor(r, dtype=torch.float32)
            d  = torch.tensor(d, dtype=torch.float32)

            q_pred = policy_net(s).gather(1, a.unsqueeze(1)).squeeze()

            # Double DQN target
            with torch.no_grad():
                best_actions = torch.argmax(policy_net(ns), dim=1)
                q_next = target_net(ns).gather(1, best_actions.unsqueeze(1)).squeeze()
                q_target = r + gamma * q_next * (1 - d)

            loss = nn.MSELoss()(q_pred, q_target)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
            optimizer.step()

    if (ep + 1) % target_update == 0:
        target_net.load_state_dict(policy_net.state_dict())

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    episode_rewards.append(total_reward)
    episode_waiting.append(total_wait / steps)
    episode_util.append(total_util / steps)

    print(f"Episode {ep+1}: Reward = {total_reward:.2f}")

print("Training Completed")

Episode 1: Reward = -2282.00
Episode 2: Reward = -2009.00
Episode 3: Reward = -1796.00
Episode 4: Reward = -1807.00
Episode 5: Reward = -1744.00
Episode 6: Reward = -2170.00
Episode 7: Reward = -1746.00
Episode 8: Reward = -1459.00
Episode 9: Reward = -1716.00
Episode 10: Reward = -1689.00
Episode 11: Reward = -1974.00
Episode 12: Reward = -1599.00
Episode 13: Reward = -1848.00
Episode 14: Reward = -1746.00
Episode 15: Reward = -1519.00
Episode 16: Reward = -1575.00
Episode 17: Reward = -1853.00
Episode 18: Reward = -1722.00
Episode 19: Reward = -2063.00
Episode 20: Reward = -1882.00
Episode 21: Reward = -2319.00
Episode 22: Reward = -1071.00
Episode 23: Reward = -1892.00
Episode 24: Reward = -1782.00
Episode 25: Reward = -1706.00
Episode 26: Reward = -1899.00
Episode 27: Reward = -1752.00
Episode 28: Reward = -1458.00
Episode 29: Reward = -1850.00
Episode 30: Reward = -1579.00
Episode 31: Reward = -1744.00
Episode 32: Reward = -1307.00
Episode 33: Reward = -1757.00
Episode 34: Reward 

In [7]:
results = pd.DataFrame({
    "episode": range(1, episodes + 1),
    "reward": episode_rewards,
    "waiting_time": episode_waiting,
    "resource_util": episode_util
})

results.to_csv("ddqn_training_log.csv", index=False)

os.makedirs("models", exist_ok=True)
torch.save(policy_net.state_dict(), "models/ddqn_model.pt")

In [8]:
plt.figure()
plt.plot(episode_rewards)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("DDQN Training Curve")
plt.savefig("ddqn_reward_curve.png")
plt.close()

In [9]:
df = pd.read_csv("ddqn_training_log.csv")

avg_reward = df["reward"].tail(20).mean()
avg_wait = df["waiting_time"].tail(20).mean()
avg_util = df["resource_util"].tail(20).mean()

print("DDQN Final Performance:")
print("Avg Reward:", avg_reward)
print("Avg Waiting Time:", avg_wait)
print("Avg Resource Util:", avg_util)

DDQN Final Performance:
Avg Reward: -475.15
Avg Waiting Time: 54.534000000000006
Avg Resource Util: 79.33399999999997


In [10]:
def convergence_episode(rewards):
    for i in range(20, len(rewards)):
        if abs(rewards[i] - rewards[i-10]) < 5:
            return i
    return len(rewards)

conv_ep = convergence_episode(episode_rewards)

pd.DataFrame({
    "convergence_episode": [conv_ep]
}).to_csv("ddqn_convergence.csv", index=False)

performance_row = pd.DataFrame({
    "Algorithm": ["DDQN"],
    "Avg Reward": [avg_reward],
    "Waiting Time": [avg_wait],
    "Resource Utilization": [avg_util],
    "Convergence Episode": [conv_ep]
})

performance_row.to_csv("ddqn_performance_row.csv", index=False)

print("DDQN experiment completed.")

DDQN experiment completed.
